# 📘 Notebook 3 — Regressione Lineare: il primo modello

Finalmente entriamo nel cuore del Machine Learning! Iniziamo con il modello **più semplice e più importante**: la **Regressione Lineare**.

## 🎯 Cosa imparerai qui
1. Cos'è la **regressione** (e perché si chiama così)
2. L'idea della **retta dei minimi quadrati**
3. Implementazione **da zero** (così capisci cosa succede sotto)
4. Implementazione con **scikit-learn** (come si fa nella pratica)
5. Come **valutare** un modello di regressione
6. **Quando usare** la regressione lineare e quando no

## 🤔 Quando usare la regressione lineare?
Quando devi **prevedere un numero** (non una categoria) e:
- vuoi un modello **veloce e semplice** da capire
- la relazione tra input e output è **approssimativamente lineare** (a linea retta)
- hai bisogno di **interpretare** il modello (capire quali variabili contano di più)

Esempi reali: prevedere il prezzo di una casa, le vendite di un negozio, il consumo di carburante di un'auto.

## 1️⃣ L'idea: tracciare la "miglior retta"

Immagina di avere dei punti su un grafico (es: ore di studio vs voto all'esame). 
La **regressione lineare** cerca **la retta che meglio si adatta** ai punti:

$$ y = w \cdot x + b $$

dove:
- $y$ = valore da prevedere (il voto)
- $x$ = caratteristica di input (le ore di studio)
- $w$ = **peso** (la pendenza della retta) — quanto un'ora in più alza il voto
- $b$ = **bias** (l'intercetta) — il valore quando $x = 0$

Il modello deve **imparare i valori giusti di $w$ e $b$** dai dati.

![minm error](Media/03/minmerror.png)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generiamo dati "finti" ma realistici: ore di studio vs voto
np.random.seed(42)
ore_studio = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
# Il voto vero segue la regola: voto ≈ 2 * ore + 4 + un po' di rumore
voto = 2 * ore_studio + 4 + np.random.normal(0, 1.5, size=10)
voto = np.clip(voto, 0, 30)   # limitiamo tra 0 e 30 (voto universitario italiano)

# Visualizziamo
plt.figure(figsize=(8, 5))
plt.scatter(ore_studio, voto, color='blue', s=80, label='Studenti')
plt.xlabel('Ore di studio')
plt.ylabel('Voto')
plt.title('Relazione tra ore di studio e voto')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# 💡 Si vede chiaramente un trend lineare: più studi, più voto.
# Ma quale retta è la "migliore"?

## 2️⃣ Cosa significa "miglior retta"? La funzione di costo

La "miglior retta" è quella che **passa più vicino possibile** a tutti i punti.
Ma come misuriamo "vicino"?

Per ogni punto calcoliamo l'**errore** (la distanza verticale dalla retta), poi lo **eleviamo al quadrato** (per non avere errori negativi che si cancellano), e infine facciamo la **media**.

Questa è la **MSE** (Mean Squared Error, errore quadratico medio):

$$ MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 $$

dove $\hat{y}_i$ è la previsione e $y_i$ il valore reale.


![MSE](Media/03/MSE.png)

Il modello deve **minimizzare la MSE**. Vediamolo visivamente:

In [ ]:
# Proviamo TRE rette diverse e vediamo l'errore
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rette = [(1.0, 2.0, 'Retta troppo bassa'),
         (3.0, 0.0, 'Retta troppo ripida'),
         (2.0, 4.0, 'Retta giusta')]

for i, (w, b, titolo) in enumerate(rette):
    ax = axes[i]
    previsioni = w * ore_studio + b
    mse = ((voto - previsioni) ** 2).mean()
    
    ax.scatter(ore_studio, voto, color='blue', s=60)
    ax.plot(ore_studio, previsioni, 'r-', linewidth=2)
    
    # Disegniamo le "linee di errore" verticali
    for x, y_vero, y_prev in zip(ore_studio, voto, previsioni):
        ax.plot([x, x], [y_vero, y_prev], 'g--', alpha=0.5)
    
    ax.set_title(f'{titolo}\nMSE = {mse:.2f}')
    ax.set_xlabel('Ore studio')
    ax.set_ylabel('Voto')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 La retta "giusta" ha la MSE più bassa. È quella che vogliamo trovare!

## 3️⃣ Trovare la "miglior retta" con la Discesa del Gradiente

Abbiamo visto nel Notebook 1 che la derivata ci dice in quale direzione andare per **diminuire** una funzione. Usiamola per trovare $w$ e $b$ che minimizzano la MSE!

**Algoritmo (idea):**
1. Inizia con $w$ e $b$ casuali
2. Calcola le previsioni e l'errore
3. Calcola le derivate dell'errore rispetto a $w$ e $b$
4. Aggiusta $w$ e $b$ nella direzione opposta
5. Ripeti finché l'errore non smette di diminuire

In [ ]:
# Implementazione DA ZERO della regressione lineare

# Iniziamo con valori casuali
w = 0.0   # peso iniziale
b = 0.0   # bias iniziale
learning_rate = 0.01   # quanto è grande ogni "passo" di aggiornamento
n = len(ore_studio)

# Salviamo la storia per disegnare poi un grafico
storia_mse = []

# Eseguiamo 1000 iterazioni ("epoche")
for epoca in range(1000):
    # 1. Calcoliamo le previsioni con i valori attuali di w e b
    previsioni = w * ore_studio + b
    
    # 2. Calcoliamo l'errore (MSE)
    errore = previsioni - voto
    mse = (errore ** 2).mean()
    storia_mse.append(mse)
    
    # 3. Calcoliamo le derivate (gradienti) di MSE rispetto a w e b
    # Formule risultanti dalla matematica delle derivate:
    grad_w = (2/n) * (errore * ore_studio).sum()
    grad_b = (2/n) * errore.sum()
    
    # 4. Aggiorniamo w e b nella direzione OPPOSTA al gradiente
    w = w - learning_rate * grad_w
    b = b - learning_rate * grad_b
    
    # Stampiamo ogni 100 epoche
    if epoca % 100 == 0:
        print(f"Epoca {epoca:4d} | MSE = {mse:6.3f} | w = {w:.3f}, b = {b:.3f}")

print(f"\n✅ Risultato finale: w = {w:.3f}, b = {b:.3f}")
print(f"   La 'regola' imparata è: voto ≈ {w:.2f} × ore_studio + {b:.2f}")

In [ ]:
# Visualizziamo come è diminuita la MSE durante l'addestramento
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(storia_mse)
plt.title('MSE durante l\'addestramento')
plt.xlabel('Epoca')
plt.ylabel('MSE')
plt.grid(True, alpha=0.3)

# E la retta finale sui dati
plt.subplot(1, 2, 2)
plt.scatter(ore_studio, voto, color='blue', s=60, label='Dati')
x_range = np.array([0, 11])
plt.plot(x_range, w * x_range + b, 'r-', linewidth=2, label=f'Retta: y = {w:.2f}x + {b:.2f}')
plt.xlabel('Ore di studio')
plt.ylabel('Voto')
plt.title('Modello finale')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 🔧 PROVA TU: cambia learning_rate a 0.1 (troppo grande) o a 0.0001 (troppo piccolo)
# e guarda cosa succede alla curva di MSE!

## 4️⃣ La stessa cosa con scikit-learn (la versione professionale)

Implementare da zero serve a **capire**. Nella pratica si usa scikit-learn, che è veloce, testato e include trucchi avanzati.

In [ ]:
from sklearn.linear_model import LinearRegression

# scikit-learn vuole X come matrice 2D (anche se ha solo 1 colonna)
X = ore_studio.reshape(-1, 1)   # da [1,2,3,...] a [[1],[2],[3],...]
y = voto

# 1. Creiamo il modello
modello = LinearRegression()

# 2. Lo addestriamo (.fit) → questa singola riga fa quello che abbiamo fatto a mano!
modello.fit(X, y)

# 3. Vediamo i parametri imparati
print(f"Coefficiente (w):  {modello.coef_[0]:.3f}")
print(f"Intercetta  (b):   {modello.intercept_:.3f}")

# 4. Facciamo previsioni
ore_test = np.array([[3.5], [7.5], [12]])
previsioni_test = modello.predict(ore_test)
for ore, voto_prev in zip(ore_test.flatten(), previsioni_test):
    print(f"   {ore} ore di studio → voto previsto: {voto_prev:.2f}")

## 5️⃣ Come valutare un modello di regressione

Tre metriche fondamentali:

| Metrica | Cosa misura | Buona se… |
|---|---|---|
| **MAE** | Errore medio assoluto | Più bassa possibile (stessa unità di y) |
| **MSE / RMSE** | Errore quadratico medio | Più bassa (penalizza errori grandi) |
| **R²** | % di varianza spiegata | Più vicina a 1 = migliore (1 = perfetto, 0 = inutile) |

![R2](Media/03/r2.png)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Previsioni del modello sui dati di training
y_pred = modello.predict(X)

# Calcoliamo le metriche
mae  = mean_absolute_error(y, y_pred)
mse  = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)   # RMSE = radice di MSE, ha la stessa unità del target
r2   = r2_score(y, y_pred)

print(f"MAE  (errore medio): {mae:.3f}  punti di voto")
print(f"RMSE (errore tipico): {rmse:.3f}  punti di voto")
print(f"R²   (qualità):       {r2:.3f}")

if r2 > 0.8:
    print("\n✅ Ottimo modello!")
elif r2 > 0.5:
    print("\n👍 Modello discreto.")
else:
    print("\n⚠️ Modello debole, serve qualcosa di meglio.")

## 6️⃣ Esempio pratico: prezzo case con più variabili (Regressione Multipla)

Nei casi reali abbiamo **molte caratteristiche**, non solo una. La formula diventa:

$$ y = w_1 x_1 + w_2 x_2 + w_3 x_3 + ... + b $$

Si chiama **Regressione Lineare Multipla**. L'idea è la stessa, solo con più pesi.

![gradient ](Media/03/gradient.png)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generiamo un dataset più ricco: 200 case
np.random.seed(42)
n_case = 200
superficie = np.random.uniform(40, 200, n_case)
stanze     = np.random.randint(1, 6, n_case)
anno       = np.random.randint(1950, 2024, n_case)
distanza_centro = np.random.uniform(0, 20, n_case)   # km dal centro

# Il prezzo "vero" segue una formula lineare + un po' di rumore
prezzo = (
    3000 * superficie       # +3000 € per ogni mq
    + 15000 * stanze        # +15000 € per stanza
    + 500 * (anno - 1950)   # +500 € per ogni anno più recente
    - 8000 * distanza_centro  # -8000 € per ogni km dal centro
    + np.random.normal(0, 25000, n_case)   # rumore (variabilità reale)
)

df = pd.DataFrame({
    'superficie': superficie,
    'stanze': stanze,
    'anno': anno,
    'distanza_centro': distanza_centro,
    'prezzo': prezzo
})
df.head()

In [ ]:
# Pipeline completa: split, scaling, training, valutazione

# 1. Separiamo X (caratteristiche) e y (target)
X = df.drop('prezzo', axis=1)
y = df['prezzo']

# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Scaling — IMPORTANTE: il fit() solo sui dati di training!
# Altrimenti "baromo" informazioni dal test set (data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # usa media/std calcolate sul training

# 4. Addestriamo il modello
modello = LinearRegression()
modello.fit(X_train_scaled, y_train)

# 5. Previsioni
y_pred_train = modello.predict(X_train_scaled)
y_pred_test  = modello.predict(X_test_scaled)

# 6. Valutazione
print("📊 PERFORMANCE DEL MODELLO")
print(f"R² training: {r2_score(y_train, y_pred_train):.3f}")
print(f"R² test:     {r2_score(y_test,  y_pred_test):.3f}")
print(f"RMSE test:   {np.sqrt(mean_squared_error(y_test, y_pred_test)):,.0f} €")

print("\n📋 IMPORTANZA DELLE CARATTERISTICHE (coefficienti)")
for nome, coef in zip(X.columns, modello.coef_):
    print(f"  {nome:20s}: {coef:>12,.0f}")

# 💡 NOTA: poiché abbiamo usato StandardScaler, i coefficienti sono confrontabili!
# Il valore assoluto più grande indica la caratteristica più "importante".
# Senza scaling, i coefficienti dipenderebbero dalle scale delle variabili.

In [ ]:
# Visualizziamo: previsioni vs valori reali
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_test, alpha=0.6)

# Linea della perfezione: dove y_pred = y_vero
lim = [y_test.min(), y_test.max()]
plt.plot(lim, lim, 'r--', label='Previsione perfetta')

plt.xlabel('Prezzo reale (€)')
plt.ylabel('Prezzo previsto (€)')
plt.title('Previsioni vs Realtà — Regressione Lineare')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 💡 Più i punti sono vicini alla linea rossa, migliore è il modello.

## 7️⃣ Limiti della Regressione Lineare (e quando NON usarla)

La regressione lineare è potente ma ha **forti assunzioni**:

1. **Linearità**: la relazione tra X e y deve essere lineare. Se è curva, fallisce.
2. **Outlier**: pochi valori estremi possono "spostare" molto la retta.
3. **Variabili correlate** (multicollinearità): se due caratteristiche dicono la stessa cosa, i coefficienti diventano instabili.

Vediamo un esempio in cui **fallisce**:

In [ ]:
# Generiamo dati con una relazione NON lineare (parabola)
np.random.seed(0)
x_curvo = np.linspace(-5, 5, 50)
y_curvo = x_curvo**2 + np.random.normal(0, 2, 50)

# Proviamo con la regressione lineare
X_curvo = x_curvo.reshape(-1, 1)
modello_lin = LinearRegression().fit(X_curvo, y_curvo)
y_pred_lin = modello_lin.predict(X_curvo)

plt.figure(figsize=(8, 5))
plt.scatter(x_curvo, y_curvo, alpha=0.6, label='Dati reali')
plt.plot(x_curvo, y_pred_lin, 'r-', linewidth=2, label='Regressione lineare')
plt.title(f'La regressione lineare NON funziona qui!\nR² = {r2_score(y_curvo, y_pred_lin):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 💡 R² è bassissimo. Per dati così serve un modello diverso
# (es: regressione polinomiale, alberi decisionali, ecc.)

## 🎓 Riepilogo del Notebook 3

**La Regressione Lineare** trova la "miglior retta" che si adatta ai dati, **minimizzando** la MSE tramite **discesa del gradiente**.

### ✅ Quando usarla
- Quando devi prevedere un numero
- Quando la relazione è (circa) lineare
- Quando vuoi un modello veloce e interpretabile
- Come **baseline** prima di provare modelli più complessi

### ❌ Quando NON usarla
- Per problemi di **classificazione** ("spam o non spam") → meglio Regressione Logistica (Notebook 4)
- Per relazioni **non lineari** complesse → meglio Random Forest, Gradient Boosting…
- Con tanti outlier → ne risente molto

### 🔑 Concetti chiave da portare via
- **Funzione di costo (MSE)**: misura quanto sbagliamo
- **Discesa del gradiente**: come il modello impara
- **Train/Test split**: per valutare onestamente
- **R², MAE, RMSE**: per misurare le performance

👉 **Prossimo notebook (04)**: la **Regressione Logistica** — quando il target non è un numero ma una **categoria** (sì/no, spam/non spam, gatto/cane).